# Create RAC (Resource Allocation Competition) Compute Allocation Awards

Creates awards for the **in-kind** compute/storage allocations of Canada's national advanced research
computing (ARC) platform, awarded through the annual **Resource Allocation Competition (RAC)**:
CPU core-years, GPU/RGU-years, project/nearline/dCache storage and cloud vCPUs/vGPUs, one row per
awarded application. Streams: Resources for Research Groups (RRG), RRG Fast Track, and Research
Platforms and Portals (RPP, multi-year).

**Source:** the public Google Sheet "List of Resource Allocation Competition <year> Awards" linked
from each year's results page / report (RAC 2021-2026, 4,068 sheet rows -> **3,847 awards**,
local run 2026-10-01). RAC 2019/2020 sheets now render only `#REF!`; RAC <= 2018 is not archived.

**Funder routing ("keep each allocation under the funder that made it"):** Compute Canada ran the
RAC through RAC 2022 (the Alliance took over on 2022-04-01), so competitions <= 2022 go to
**Compute Canada (F4320314000)**: 1,260 awards; competitions >= 2023 go to the
**Digital Research Alliance of Canada (F4320331257)**: 2,587 awards. An RPP is routed by its
application year. This is a separate provenance from the Alliance cash programmes
(`CreateAllianceCanAwards`, priority 548).

**Prerequisites:**
- Run `scripts/local/alliancecan_rac_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/alliancecan_rac/alliancecan_rac_projects.parquet` (1.4 shrink guard).

**`funder_award_id` (2.1.1):** papers cite the App ID with its stream ("RRG 4073", "RPP 772",
"RRG 1541 RAC 2021"), so we ship `RRG <id>` / `RPP <id>` (Fast Track = an RRG pathway, cited as
RRG). RPP ids persist across an RPP's years (one award, allocations listed per year); RRG ids are
annual and get recycled years later for other PIs, always across the two funders (the script raises
on any within-funder collision). Exact collapse onto existing citation stubs: 4 (RPP 772, RRG 3822,
RRG 4448, RRG 4657, all under F4320331257); most stubs use other spellings ("RRG-4802", "RRG#5303",
CCDB RAP ids like "xvj-290-01").

**Amounts / dates:** in-kind, so `amount`/`currency` are NULL (the reports publish only per-unit
"financial value" rates, never a per-award value); the allocations are spelled out in `description`.
`funding_type = 'grant'` (same as NWO's computing-time awards, the only existing in-kind compute
allocations in `openalex_awards`). `start_year` = first RAC year; `end_year` = last RAC year + 1
(allocation periods run April-March, published as e.g. "the 2023-2024 allocation period").
No start/end dates are published.

**lead_investigator:** the PI / applicant (split in Python, runbook 2.4.1), institution as
affiliation (country CA). One 2024 row carries a number in the applicant column -> PI NULL.

**Funders (Path A, F4320* Crossref-registered):** 4320314000 Compute Canada (ror 03ty8yr27);
4320331257 Alliance de recherche numerique du Canada (ror 010r6td27).

**Priority:** `553` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alliancecan_rac_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/alliancecan_rac/alliancecan_rac_projects.parquet`;

In [ ]:
%sql
SELECT funder_org, funder_id, funder_scheme, COUNT(*) as n
FROM openalex.awards.alliancecan_rac_raw
GROUP BY 1, 2, 3 ORDER BY 1, 3;

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.alliancecan_rac_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.alliancecan_rac_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
Both funder ids are Crossref-registered (F4320*), so each MUST resolve to exactly 1 row in
`openalex.funders.funders`. If not, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id IN (4320314000, 4320331257);

In [ ]:
%sql
SELECT
    assert_true(COUNT_IF(funder_id = 4320314000) = 1, 'F4320314000 (Compute Canada) must have exactly one row') AS cc_ok,
    assert_true(COUNT_IF(funder_id = 4320331257) = 1, 'F4320331257 (the Alliance) must have exactly one row') AS alliance_ok
FROM openalex.funders.funders
WHERE funder_id IN (4320314000, 4320331257);

## Step 2: Create RAC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alliancecan_rac_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id IN (4320314000, 4320331257)
),

g AS (
    SELECT
        r.*,
        TRY_CAST(r.funder_id AS BIGINT) AS funder_id_num,
        TRIM(r.funder_award_id) AS award_id
    FROM openalex.awards.alliancecan_rac_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_id_num,
        award_id,
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'CA',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead_investigator
    FROM g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_id)))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('RAC allocation ', g.award_id)) as display_name,
    CONCAT_WS(' ',
        'In-kind allocation of national advanced research computing resources (no monetary award).',
        CONCAT(g.allocation_text, '.'),
        CASE WHEN NULLIF(TRIM(g.summary), '') IS NOT NULL THEN CONCAT('Project summary: ', TRIM(g.summary)) END
    ) as description,
    f.funder_id,
    g.award_id as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,
    'alliancecan_rac' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.first_year AS INT) as start_year,
    TRY_CAST(g.last_year AS INT) + 1 as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN l.lead_investigator IS NOT NULL THEN array(l.lead_investigator) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
JOIN lead l ON l.funder_id_num = g.funder_id_num AND l.award_id = g.award_id
JOIN src_funder f ON f.funder_id = g.funder_id_num;

In [ ]:
%sql
-- Shape check (2.1.1): "RRG <id>" / "RPP <id>" only; ids unique; only the two RAC funders; no rows lost.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^(RRG|RPP) [0-9]+$') = 0,
                'funder_award_id is not an RRG/RPP <App ID> reference') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok,
    assert_true(COUNT_IF(funder_id NOT IN (4320314000, 4320331257)) = 0, 'unexpected funder') AS funder_ok,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.alliancecan_rac_raw), 'rows lost in transform') AS complete_ok
FROM openalex.awards.alliancecan_rac_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alliancecan_rac' AND priority = 553;

-- Insert into openalex_awards_raw with priority.
-- Priority 553: direct-from-funder ingest of the published RAC award lists (in-kind compute allocations).
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    553 as priority
FROM openalex.awards.alliancecan_rac_awards;

## Verification Queries

In [ ]:
%sql
SELECT funder_id, funder.display_name, COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.alliancecan_rac_awards
GROUP BY 1, 2;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name,
       LEFT(description, 300) AS description_head
FROM openalex.awards.alliancecan_rac_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 (RRG is annual, so a PI can hold up to one award per RAC year).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.alliancecan_rac_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.alliancecan_rac_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (in-kind: pct_amount 0 by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.alliancecan_rac_awards;

In [ ]:
%sql
SELECT start_year, funder_id, COUNT(*) AS n
FROM openalex.awards.alliancecan_rac_awards
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, funder_id, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alliancecan_rac'
GROUP BY provenance, priority, funder_id;